In [ ]:
import numpy as np

def downsample_masked_mean(row, N, max_range=20.0, min_valid_frac=0.1):
    """
    row: shape (2760,) float32 LiDAR ranges over 270°
    N: target bins (128, 64, 32)
    Returns (y, valid_frac): both length N
    """
    x = row.copy()
    valid = (x > 0.0) & (x <= max_range) & np.isfinite(x)
    x[~valid] = 0.0
    w = valid.astype(np.float32)

    M = len(x)             # 2760
    # Bin j covers [a,b) in source index space
    edges = np.linspace(0.0, M, N+1)
    y = np.empty(N, dtype=np.float32)
    vf = np.empty(N, dtype=np.float32)

    # Cumsums let us get sums over [i:j) fast; we also handle fractional edges.
    sx  = np.concatenate(([0.0], np.cumsum(x)))
    sw  = np.concatenate(([0.0], np.cumsum(w)))

    # helper: integral up to fractional index t assuming piecewise-constant per sample
    def integral_at(t, s, arr):
        i = int(np.floor(t))
        frac = t - i
        i = min(i, M-1)
        # sum of full samples + fractional part of current sample
        return s[i] + frac * arr[i]

    for j in range(N):
        a, b = edges[j], edges[j+1]
        sum_x = integral_at(b, sx, x) - integral_at(a, sx, x)
        sum_w = integral_at(b, sw, w) - integral_at(a, sw, w)
        vf[j] = sum_w / (b - a)             # valid fraction within this new bin
        y[j]  = (sum_x / sum_w) if sum_w > 0 else np.nan

    # Optional: invalidate bins with too little valid support
    y[vf < min_valid_frac] = np.nan
    return y, vf
